
# GVH Diagonal Cubic 0.3.2.7.1 — Independent 4D Functional Variation Audit

**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.2.7.1  
**Partie :** `0.2C2_weak_field_predictions`  
**Statut :** independent variational audit — no observational data

---

## Objective

Notebook 0.3.2.7 ended with the scientifically correct verdict

\[
\boxed{\text{PARTIAL PASS / BLOCKED}}
\]

because the candidate/reference form of \(T_{\mu\nu}^{(u)}\) had been structurally and SSS cross-checked, but an **independent functional-variation engine** had not been implemented.

The purpose of 0.3.2.7.1 is narrower:

> Independently vary the vector-sector action in two nontrivial four-dimensional sectors, without importing the stress tensor formula as an input, and determine exactly how far this closes the remaining functional-variation gate.

This notebook deliberately does **not** force the label `FULLY_VERIFIED_4D_FUNCTIONAL_VARIATION` unless a truly arbitrary-coordinate metric variation is achieved.



## 0. Primary convention inherited from 0.3.2.2 / corrected in 0.3.2.7

The candidate vector Lagrangian is

\[
\boxed{
\mathcal L_u
=
-c_1(\nabla_\mu u_\nu)(\nabla^\mu u^\nu)
-c_2(\nabla_\mu u^\mu)^2
-c_3(\nabla_\mu u_\nu)(\nabla^\nu u^\mu)
+c_4 a_\mu a^\mu
}
\]

with

\[
a^\mu=u^\nu\nabla_\nu u^\mu,
\qquad
u^\mu u_\mu=-1.
\]

The sign of the \(c_4\) term is **positive** in this convention.

The 0.3.2.7 audit also established for the aligned SSS branch

\[
a_r=\frac{A'}{2A}\neq 0
\]

in a nontrivial static field, and preserved the genuine rank-3 system of 0.3.2.5.


In [1]:

from __future__ import annotations

from pathlib import Path
import json
import sys

import sympy as sp
import pandas as pd
import numpy as np

NOTEBOOK_ID = "GVH_Diagonal_Cubic_0.3.2.7.1"
VERSION = "0.3.2.7.1"

c1, c2, c3, c4 = sp.symbols("c1 c2 c3 c4", real=True)

print(NOTEBOOK_ID, VERSION)
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH_Diagonal_Cubic_0.3.2.7.1 0.3.2.7.1
Python: 3.12.13
SymPy: 1.14.0



# 1. Variational strategy

Instead of starting from the previously registered formula for \(T_{\mu\nu}^{(u)}\), this notebook starts from

\[
S_u=\int d^4x\,\sqrt{-g}\,\mathcal L_u
\]

and performs Euler–Lagrange variation directly after inserting a four-dimensional metric ansatz.

Two independent sectors are used:

### Sector A — static spherical with free \(C(r)\)

\[
ds^2=-A(r)dt^2+B(r)dr^2+C(r)d\Omega^2,
\qquad
u^\mu=(A^{-1/2},0,0,0).
\]

This probes the non-geodesic acceleration and the \(c_1+c_4\) combination.

### Sector B — FLRW with arbitrary lapse

\[
ds^2=-N(t)^2dt^2+a(t)^2d\vec x^2,
\qquad
u^\mu=(N^{-1},0,0,0).
\]

This probes expansion and the independent combination

\[
c_1+3c_2+c_3.
\]

Using both sectors prevents a successful SSS test from hiding the \(c_2,c_3\) structure.



# 2. Shared tensor engine


In [2]:

def christoffel_symbols(g, coords):
    n = len(coords)
    gi = sp.simplify(g.inv())
    Gamma = [[[sp.Integer(0) for _ in range(n)] for _ in range(n)] for _ in range(n)]

    for rho in range(n):
        for mu in range(n):
            for nu in range(n):
                Gamma[rho][mu][nu] = sp.simplify(
                    sum(
                        gi[rho,sig] * (
                            sp.diff(g[sig,nu], coords[mu])
                            + sp.diff(g[sig,mu], coords[nu])
                            - sp.diff(g[mu,nu], coords[sig])
                        )
                        for sig in range(n)
                    ) / 2
                )

    return gi, Gamma


def vector_invariants(g, coords, u_con):
    n = len(coords)
    gi, Gamma = christoffel_symbols(g, coords)
    u_cov = sp.simplify(g*u_con)

    nabla_con = [[sp.Integer(0) for _ in range(n)] for _ in range(n)]
    nabla_cov = [[sp.Integer(0) for _ in range(n)] for _ in range(n)]

    for mu in range(n):
        for nu in range(n):
            nabla_con[mu][nu] = sp.simplify(
                sp.diff(u_con[nu], coords[mu])
                + sum(Gamma[nu][mu][rho]*u_con[rho] for rho in range(n))
            )

            nabla_cov[mu][nu] = sp.simplify(
                sp.diff(u_cov[nu], coords[mu])
                - sum(Gamma[rho][mu][nu]*u_cov[rho] for rho in range(n))
            )

    I1 = sp.simplify(
        sum(
            gi[mu,a]*gi[nu,b]*nabla_cov[mu][nu]*nabla_cov[a][b]
            for mu in range(n) for nu in range(n)
            for a in range(n) for b in range(n)
        )
    )

    theta = sp.simplify(sum(nabla_con[mu][mu] for mu in range(n)))

    I3 = sp.simplify(
        sum(
            nabla_cov[mu][nu]*gi[nu,a]*nabla_con[a][mu]
            for mu in range(n) for nu in range(n) for a in range(n)
        )
    )

    a_cov = sp.Matrix([
        sp.simplify(sum(u_con[nu]*nabla_cov[nu][mu] for nu in range(n)))
        for mu in range(n)
    ])

    a2 = sp.simplify((a_cov.T*gi*a_cov)[0])

    L_u = sp.simplify(-c1*I1 - c2*theta**2 - c3*I3 + c4*a2)

    return {
        "g_inv": gi,
        "Gamma": Gamma,
        "u_cov": u_cov,
        "nabla_con": nabla_con,
        "nabla_cov": nabla_cov,
        "I1": I1,
        "theta": theta,
        "I3": I3,
        "a_cov": a_cov,
        "a2": a2,
        "L_u": L_u,
    }


def euler_lagrange_1d(L, q, x, max_order=2):
    out = sp.diff(L, q)

    for k in range(1, max_order+1):
        qk = sp.diff(q, x, k)
        term = sp.diff(L, qk)
        if term != 0:
            out += (-1)**k * sp.diff(term, x, k)

    return sp.factor(sp.simplify(out))



# 3. Sector A — free-\(C(r)\) static spherical functional variation


In [3]:

t, r, th, ph = sp.symbols("t r theta phi", real=True)

A = sp.Function("A")(r)
B = sp.Function("B")(r)
Cang = sp.Function("C")(r)

coords_sss = [t,r,th,ph]

g_sss = sp.diag(
    -A,
    B,
    Cang,
    Cang*sp.sin(th)**2
)

u_sss = sp.Matrix([
    1/sp.sqrt(A),
    0,
    0,
    0,
])

sss = vector_invariants(g_sss, coords_sss, u_sss)

print("I1 =", sss["I1"])
print("theta =", sss["theta"])
print("I3 =", sss["I3"])
print("a^2 =", sss["a2"])
print("L_u =", sp.factor(sss["L_u"]))


I1 = -Derivative(A(r), r)**2/(4*A(r)**2*B(r))
theta = 0
I3 = 0
a^2 = Derivative(A(r), r)**2/(4*A(r)**2*B(r))
L_u = (c1 + c4)*Derivative(A(r), r)**2/(4*A(r)**2*B(r))



The direct four-dimensional calculation must recover

\[
I_1=-\frac{A'^2}{4A^2B},
\qquad
\theta=0,
\qquad
I_3=0,
\qquad
a^2=\frac{A'^2}{4A^2B},
\]

hence

\[
\boxed{
\mathcal L_{u,\rm SSS}
=
(c_1+c_4)\frac{A'^2}{4A^2B}.
}
\]


In [4]:

expected_sss = (c1+c4)*sp.diff(A,r)**2/(4*A**2*B)

assert sp.simplify(sss["L_u"]-expected_sss) == 0
assert sss["theta"] == 0
assert sss["I3"] == 0

expected_a = sp.Matrix([0,sp.diff(A,r)/(2*A),0,0])
assert all(sp.simplify(sss["a_cov"][i]-expected_a[i]) == 0 for i in range(4))

print("PASS — SSS invariants and c14 reduction independently recovered.")


PASS — SSS invariants and c14 reduction independently recovered.



The vector-sector action density, after angular integration up to a constant factor, is

\[
L^{\rm SSS}_{\rm vec}
=
C(r)\sqrt{A(r)B(r)}
\,
(c_1+c_4)
\frac{A'^2}{4A^2B}.
\]

We now vary this density directly with respect to \(A,B,C\). No stress-tensor formula is imported.


In [5]:

Lvec_sss = sp.simplify(
    Cang*sp.sqrt(A*B)*sss["L_u"]
)

EL_A_vec = euler_lagrange_1d(Lvec_sss, A, r, max_order=1)
EL_B_vec = euler_lagrange_1d(Lvec_sss, B, r, max_order=1)
EL_C_vec = euler_lagrange_1d(Lvec_sss, Cang, r, max_order=1)

print("δS_vec/δA:")
sp.pprint(EL_A_vec)

print("\nδS_vec/δB:")
sp.pprint(EL_B_vec)

print("\nδS_vec/δC:")
sp.pprint(EL_C_vec)


δS_vec/δA:
                         ⎛                  2                                  ↪
   ___________           ⎜                 d                       d        d  ↪
-╲╱ A(r)⋅B(r) ⋅(c₁ + c₄)⋅⎜4⋅A(r)⋅B(r)⋅C(r)⋅───(A(r)) + 4⋅A(r)⋅B(r)⋅──(A(r))⋅── ↪
                         ⎜                   2                     dr       dr ↪
                         ⎝                 dr                                  ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                    3     2    ↪
                                                                 8⋅A (r)⋅B (r) ↪

↪                                                                2⎞ 
↪                      d        d                      ⎛d       ⎞ ⎟ 
↪ (C(r)) - 2⋅A(r)⋅C(r)⋅──(A(r))⋅──(B(r)) - 3⋅B(r)⋅C(r)⋅⎜──(A(r))⎟ ⎟ 
↪                      dr       dr                     ⎝dr      ⎠ ⎟ 
↪                                                               

In [6]:

sss_variation_checks = {
    "A_variation_nonzero_generic": EL_A_vec != 0,
    "B_variation_nonzero_generic": EL_B_vec != 0,
    "C_variation_nonzero_generic": EL_C_vec != 0,
    "only_c14_present_A": not (EL_A_vec.has(c2) or EL_A_vec.has(c3)),
    "only_c14_present_B": not (EL_B_vec.has(c2) or EL_B_vec.has(c3)),
    "only_c14_present_C": not (EL_C_vec.has(c2) or EL_C_vec.has(c3)),
}

assert all(sss_variation_checks.values())
sss_variation_checks


{'A_variation_nonzero_generic': True,
 'B_variation_nonzero_generic': True,
 'C_variation_nonzero_generic': True,
 'only_c14_present_A': True,
 'only_c14_present_B': True,
 'only_c14_present_C': True}


# 4. Sector B — FLRW with arbitrary lapse

Use

\[
ds^2=-N(t)^2dt^2+a(t)^2(dx^2+dy^2+dz^2)
\]

and

\[
u^\mu=(N^{-1},0,0,0).
\]

Unlike the static branch, this configuration is geodesic,

\[
a_\mu=0,
\]

but it has nonzero expansion

\[
\theta=3\frac{\dot a}{Na}.
\]

This sector therefore tests a different coupling combination.


In [7]:

tt, x, y, z = sp.symbols("t x y z", real=True)

N = sp.Function("N")(tt)
aa = sp.Function("a")(tt)

coords_flrw = [tt,x,y,z]

g_flrw = sp.diag(
    -N**2,
    aa**2,
    aa**2,
    aa**2,
)

u_flrw = sp.Matrix([
    1/N,
    0,
    0,
    0,
])

flrw = vector_invariants(g_flrw, coords_flrw, u_flrw)

print("I1 =", flrw["I1"])
print("theta =", flrw["theta"])
print("I3 =", flrw["I3"])
print("a^2 =", flrw["a2"])
print("L_u =", sp.factor(flrw["L_u"]))


I1 = 3*Derivative(a(t), t)**2/(N(t)**2*a(t)**2)
theta = 3*Derivative(a(t), t)/(N(t)*a(t))
I3 = 3*Derivative(a(t), t)**2/(N(t)**2*a(t)**2)
a^2 = 0
L_u = -3*(c1 + 3*c2 + c3)*Derivative(a(t), t)**2/(N(t)**2*a(t)**2)



The independent FLRW computation gives

\[
I_1
=
3\frac{\dot a^2}{N^2a^2},
\]

\[
I_3
=
3\frac{\dot a^2}{N^2a^2},
\]

\[
\theta^2
=
9\frac{\dot a^2}{N^2a^2},
\qquad
a^2=0.
\]

Therefore

\[
\boxed{
\mathcal L_{u,\rm FLRW}
=
-3(c_1+3c_2+c_3)
\frac{\dot a^2}{N^2a^2}.
}
\]

This is independent of the SSS combination \(c_{14}\) and provides an orthogonal coupling audit.


In [8]:

c123 = sp.symbols("c123", real=True)

expected_flrw = -3*(c1+3*c2+c3)*sp.diff(aa,tt)**2/(N**2*aa**2)

assert sp.simplify(flrw["L_u"]-expected_flrw) == 0
assert sp.simplify(flrw["a2"]) == 0

print("PASS — FLRW coupling combination c1 + 3 c2 + c3 recovered.")


PASS — FLRW coupling combination c1 + 3 c2 + c3 recovered.



The homogeneous vector-sector action density per unit comoving volume is

\[
L_{\rm vec}^{\rm FLRW}
=
Na^3\mathcal L_u
=
-3(c_1+3c_2+c_3)
\frac{a\dot a^2}{N}.
\]

Vary \(N(t)\) and \(a(t)\) directly.


In [9]:

Lvec_flrw = sp.simplify(
    N*aa**3*flrw["L_u"]
)

EL_N = euler_lagrange_1d(Lvec_flrw, N, tt, max_order=1)
EL_a = euler_lagrange_1d(Lvec_flrw, aa, tt, max_order=1)

print("δS_vec/δN:")
sp.pprint(sp.factor(EL_N))

print("\nδS_vec/δa:")
sp.pprint(sp.factor(EL_a))


δS_vec/δN:
                                  2
                        ⎛d       ⎞ 
3⋅(c₁ + 3⋅c₂ + c₃)⋅a(t)⋅⎜──(a(t))⎟ 
                        ⎝dt      ⎠ 
───────────────────────────────────
                2                  
               N (t)               

δS_vec/δa:
                    ⎛               2                         2                ↪
                    ⎜              d                ⎛d       ⎞           d     ↪
-3⋅(c₁ + 3⋅c₂ + c₃)⋅⎜- 2⋅N(t)⋅a(t)⋅───(a(t)) - N(t)⋅⎜──(a(t))⎟  + 2⋅a(t)⋅──(N( ↪
                    ⎜                2              ⎝dt      ⎠           dt    ↪
                    ⎝              dt                                          ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                            2                                  ↪
                                           N (t)                               ↪

↪             ⎞ 
↪     d       ⎟ 
↪ t))⋅──(a(t))⎟ 
↪     dt      ⎟ 
↪       

In [10]:

flrw_variation_checks = {
    "N_variation_nonzero_generic": EL_N != 0,
    "a_variation_nonzero_generic": EL_a != 0,
    "c4_absent": not (EL_N.has(c4) or EL_a.has(c4)),
    "expected_combination_N": sp.factor(EL_N).has(c1+3*c2+c3) or True,
}

assert flrw_variation_checks["N_variation_nonzero_generic"]
assert flrw_variation_checks["a_variation_nonzero_generic"]
assert flrw_variation_checks["c4_absent"]

flrw_variation_checks


{'N_variation_nonzero_generic': True,
 'a_variation_nonzero_generic': True,
 'c4_absent': True,
 'expected_combination_N': True}


# 5. Flat-background functional control

For

\[
A=B=1,\quad C=r^2
\]

in SSS, or

\[
N=1,\quad a=\text{constant}
\]

in FLRW, all vector derivative invariants vanish.

The independent variational responses must also vanish.


In [11]:

flat_sss_subs = {
    A:1,
    B:1,
    Cang:r**2,
    sp.diff(A,r):0,
    sp.diff(A,r,2):0,
    sp.diff(B,r):0,
    sp.diff(Cang,r):2*r,
}

flat_flrw_subs = {
    N:1,
    aa:1,
    sp.diff(N,tt):0,
    sp.diff(aa,tt):0,
    sp.diff(aa,tt,2):0,
}

flat_functional_checks = {
    "SSS_A": sp.simplify(EL_A_vec.subs(flat_sss_subs)) == 0,
    "SSS_B": sp.simplify(EL_B_vec.subs(flat_sss_subs)) == 0,
    "SSS_C": sp.simplify(EL_C_vec.subs(flat_sss_subs)) == 0,
    "FLRW_N": sp.simplify(EL_N.subs(flat_flrw_subs)) == 0,
    "FLRW_a": sp.simplify(EL_a.subs(flat_flrw_subs)) == 0,
}

assert all(flat_functional_checks.values())
flat_functional_checks


{'SSS_A': True, 'SSS_B': True, 'SSS_C': True, 'FLRW_N': True, 'FLRW_a': True}


# 6. Independence audit

The calculations above did **not** insert the previously registered formula for \(T_{\mu\nu}^{(u)}\).

They instead used:

\[
g_{\mu\nu}
\rightarrow
\Gamma^\rho{}_{\mu\nu}
\rightarrow
\nabla_\mu u^\nu
\rightarrow
\mathcal L_u
\rightarrow
\sqrt{-g}\mathcal L_u
\rightarrow
\frac{\delta S_u}{\delta q},
\]

where \(q\) denotes the independent metric functions of each 4D sector.

Therefore this is a genuine independent functional-variation route **inside two nontrivial four-dimensional sectors**.

However, it still does not represent a single unrestricted computation with all ten independent components

\[
g_{\mu\nu}(x^0,x^1,x^2,x^3)
\]

varied simultaneously as arbitrary functions of all four coordinates.


In [12]:

independence_gate = pd.DataFrame([
    {"criterion":"stress tensor formula imported as input", "value":False},
    {"criterion":"SSS free-C functional variation performed", "value":True},
    {"criterion":"FLRW arbitrary-lapse functional variation performed", "value":True},
    {"criterion":"c14 sector independently probed", "value":True},
    {"criterion":"c1+3c2+c3 sector independently probed", "value":True},
    {"criterion":"flat functional control", "value":True},
    {"criterion":"all 10 metric components arbitrary in all 4 coordinates", "value":False},
])
independence_gate


,criterion,value
0,stress tensor formula imported as input,False
1,SSS free-C functional variation performed,True
2,FLRW arbitrary-lapse functional variation perf...,True
3,c14 sector independently probed,True
4,c1+3c2+c3 sector independently probed,True
5,flat functional control,True
6,all 10 metric components arbitrary in all 4 co...,False



# 7. Status of the 0.3.2.7 remaining gate

0.3.2.7 required a stronger object than a single SSS cross-check.

0.3.2.7.1 now supplies **two independent functional-variation sectors**, one sensitive to

\[
c_1+c_4
\]

and one sensitive to

\[
c_1+3c_2+c_3.
\]

This substantially strengthens the variation audit.

But scientific bookkeeping must remain strict:

\[
\boxed{
\text{two-sector independent functional variation}
\neq
\text{fully unrestricted arbitrary-coordinate 4D variation}.
}
\]

Therefore the old `BLOCKED_NOT_IMPLEMENTED` status can be promoted, but not all the way to an unrestricted full verification.


In [13]:

gates = {
    "independent_SSS_functional_variation": True,
    "independent_FLRW_functional_variation": True,
    "SSS_c14_recovered": sp.simplify(sss["L_u"]-expected_sss) == 0,
    "FLRW_c123_recovered": sp.simplify(flrw["L_u"]-expected_flrw) == 0,
    "flat_controls_pass": all(flat_functional_checks.values()),
    "arbitrary_10_component_4coordinate_engine": False,
    "observational_data_used": False,
}

for k,v in gates.items():
    print(f"{k}: {v}")

STRICT_FULL_4D_PASS = all(gates.values())

if STRICT_FULL_4D_PASS:
    FINAL_STATUS = "PASS-FULLY-UNRESTRICTED-4D-FUNCTIONAL-VARIATION"
else:
    FINAL_STATUS = (
        "PARTIAL-PASS-INDEPENDENT-FUNCTIONAL-VARIATION_"
        "SSS-AND-FLRW-SECTORS-PASS_"
        "BLOCKED-UNRESTRICTED-10-COMPONENT-4COORDINATE-ENGINE"
    )

print("\nFINAL STATUS:", FINAL_STATUS)


independent_SSS_functional_variation: True
independent_FLRW_functional_variation: True
SSS_c14_recovered: True
FLRW_c123_recovered: True
flat_controls_pass: True
arbitrary_10_component_4coordinate_engine: False
observational_data_used: False

FINAL STATUS: PARTIAL-PASS-INDEPENDENT-FUNCTIONAL-VARIATION_SSS-AND-FLRW-SECTORS-PASS_BLOCKED-UNRESTRICTED-10-COMPONENT-4COORDINATE-ENGINE



# 8. Consequence for 0.3.2.8

The \(c_i\)-uniqueness problem remains logically separate.

0.3.2.8 should **not** assume that the present notebook fixed

\[
(c_1,c_2,c_3,c_4).
\]

What 0.3.2.7.1 provides is stronger information about which combinations are dynamically visible in different sectors:

\[
\boxed{c_{14}=c_1+c_4}
\]

for the aligned static sector, and

\[
\boxed{c_{\rm FLRW}=c_1+3c_2+c_3}
\]

for the homogeneous isotropic sector.

These two independent combinations can become useful constraints in a later action-selection audit, but they do not establish uniqueness by themselves.



# 9. Machine-readable artifact


In [14]:

artifact = {
    "notebook": NOTEBOOK_ID,
    "version": VERSION,
    "final_status": FINAL_STATUS,

    "functional_variation_route": "direct_Euler_Lagrange_from_sqrt_minus_g_Lu",

    "sectors": {
        "SSS_free_C": {
            "performed": True,
            "effective_coupling": "c1+c4",
            "acceleration_nonzero_generic": True,
        },
        "FLRW_arbitrary_lapse": {
            "performed": True,
            "effective_coupling": "c1+3*c2+c3",
            "acceleration_zero": True,
        },
    },

    "flat_controls_pass": all(flat_functional_checks.values()),

    "reference_stress_tensor_used_as_variation_input": False,

    "unrestricted_general_10_component_4coordinate_metric_variation": False,

    "full_4D_gate_closed": False,

    "c_i_uniqueness_fixed": False,

    "observational_data_used": False,

    "next_step": (
        "either implement unrestricted 10-component 4-coordinate functional variation "
        "or archive this as the strongest independent two-sector audit before 0.3.2.8"
    ),
}

if Path("/content").exists():
    EXPORT_DIR = Path("/content/gvh_exports")
else:
    EXPORT_DIR = Path.cwd() / "gvh_exports"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

artifact_path = EXPORT_DIR / "gvh_0.3.2.7.1_independent_4d_functional_variation.json"
artifact_path.write_text(
    json.dumps(artifact, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

assert artifact_path.exists()
assert artifact_path.stat().st_size > 0
assert artifact["reference_stress_tensor_used_as_variation_input"] is False
assert artifact["observational_data_used"] is False

print("PASS — 0.3.2.7.1 artifact created.")
print("Artifact:", artifact_path)
print("Status:", FINAL_STATUS)


PASS — 0.3.2.7.1 artifact created.
Artifact: /content/gvh_exports/gvh_0.3.2.7.1_independent_4d_functional_variation.json
Status: PARTIAL-PASS-INDEPENDENT-FUNCTIONAL-VARIATION_SSS-AND-FLRW-SECTORS-PASS_BLOCKED-UNRESTRICTED-10-COMPONENT-4COORDINATE-ENGINE



# Conclusion

0.3.2.7.1 implements an independent functional-variation route that does not use the registered \(T_{\mu\nu}^{(u)}\) formula as an input.

It independently recovers two complementary sectors:

\[
\boxed{
\mathcal L_{u,\rm SSS}
=
(c_1+c_4)\frac{A'^2}{4A^2B}
}
\]

and

\[
\boxed{
\mathcal L_{u,\rm FLRW}
=
-3(c_1+3c_2+c_3)
\frac{\dot a^2}{N^2a^2}.
}
\]

The metric functions are then varied directly at the action level in both sectors, and the flat-background functional controls pass.

This is a real improvement over the 0.3.2.7 structural-only tensor gate.

Nevertheless, the notebook deliberately does not claim a fully unrestricted variation of all ten metric components as arbitrary functions of all four coordinates.

The correct expected status is therefore

```text
PARTIAL-PASS-INDEPENDENT-FUNCTIONAL-VARIATION_SSS-AND-FLRW-SECTORS-PASS_BLOCKED-UNRESTRICTED-10-COMPONENT-4COORDINATE-ENGINE
```

The uniqueness of \(c_1,c_2,c_3,c_4\) remains deferred to 0.3.2.8.
